[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/internshipinabook/cv-internship-in-a-book/blob/main/notebooks/week2_eda_on_pixels_STARTER.ipynb)


# Week 2 -- Reading Images — EDA on Pixels (STARTER)
### The Computer Vision Internship | MediVision AI Lagos

**This week:** EDA on Oxford-IIIT Pets — class balance, colour distributions, augmentation audit

**Dataset:** Oxford-IIIT Pets (EDA)

**STARTER notebook** -- your working environment. Complete each exercise before moving on.


In [ ]:
# -- Colab/Local Setup -- run this first in Colab, skip locally -------------
import os

if not os.path.exists('requirements.txt'):
    # Clone the full course repository
    !git clone https://github.com/internshipinabook/cv-internship-in-a-book.git cv-book
    %cd cv-book
    # Install all required packages (suppress verbose output with -q)
    !pip install -r requirements.txt -q

# Create working directories
os.makedirs('outputs',  exist_ok=True)   # charts, reports, predictions
os.makedirs('models',   exist_ok=True)   # trained model checkpoints
os.makedirs('datasets', exist_ok=True)   # raw downloaded data
print('Setup complete.')


In [ ]:
# -- Reproducibility Seeds --------------------------------------------------
# Setting seeds ensures every random operation produces the same result
# on every run. Essential for: comparing runs, debugging, sharing results.

import random    # Python built-in random
import numpy as np   # NumPy random (data loading, sklearn)
import torch     # PyTorch random (neural networks, dropout)

SEED = 42        # 42 is conventional -- any integer works

random.seed(SEED)           # fix Python random() calls
np.random.seed(SEED)        # fix np.random.* calls
torch.manual_seed(SEED)     # fix PyTorch CPU operations

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)  # fix GPU operations too

# Prefer deterministic cuDNN algorithms (may slow training ~5%)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Seeds fixed: {SEED} | Device: {device}')


## Learning Objectives

By the end of this week, you will be able to:

- Compute class distribution for Oxford-IIIT Pets and identify imbalance before any modelling
- Plot channel histograms and identify the normalisation parameters your dataset requires
- Apply and evaluate five augmentation strategies and explain the tradeoff for each
- Detect and quantify label noise in the Pets dataset using cross-referencing
- Write an EDA report that a product manager could use to scope the modelling task



---

## MONDAY -- "Class Distribution — What Did the Annotators Label?"


Oxford-IIIT Pets has 37 breeds. The distribution is not uniform. Some breeds have 200 images; others have fewer than 100. Class imbalance at this level will cause the model to be systematically better at common breeds. Before training, you must know which classes are rare — and decide whether to oversample, undersample, or weight the loss.


### Exercise 2.1 -- Class distribution audit

Load all annotation files. Plot the class distribution as a horizontal bar chart. Title it as a finding: "8 of 37 Pets breeds have fewer than 100 training images." Which three classes have the fewest examples?


In [ ]:
# Exercise 2.1: Class distribution audit
# -------------------------------------------------------
# YOUR CODE HERE
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the bottom of this notebook.


### Exercise 2.2 -- Imbalance ratio

Compute the imbalance ratio (max class count / min class count). What weighted loss strategy would you apply? Write your decision before implementing it.


In [ ]:
# Exercise 2.2: Imbalance ratio
# -------------------------------------------------------
# YOUR CODE HERE
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the bottom of this notebook.


**Monday code scaffold** (read and run, then adapt in the exercise cells above):


In [ ]:
# -- Monday: Class Distribution — What Did the Annotators Label? (scaffold) --
import pandas as pd
from collections import Counter
import matplotlib.pyplot as plt

# Load the annotation file
with open("datasets/pets/annotations/list.txt") as f:
    lines = [l.strip().split() for l in f if not l.startswith("#")]
class_counts = Counter(row[1] for row in lines)
df = pd.DataFrame(class_counts.items(), columns=["class_id","count"]).sort_values("count")
df.plot(kind="barh", x="class_id", y="count", figsize=(8,12))
plt.title("Oxford Pets: samples per class — 8 classes have fewer than 100 images")


### Monday Morning Moment

*Slack — Monday, 2:30pm.*

**Ngozi Eze-Okafor:** The class distribution is worse than I expected. 8 breeds have fewer than 100 images.

**Yewande Adeyemi:** That's fine — just use class_weight in your loss function.

**Dr. Kwame Asante:** Yewande, what happens in Week 7 when we have 4× more benign patches than malignant?

**Yewande Adeyemi:** ...same solution. Weighted loss.

**Dr. Kwame Asante:** Ngozi — write down every imbalance you find this week. The pattern repeats in every medical dataset. Pets is the practice run.




---

## TUESDAY -- "Channel Histograms and Normalisation Parameters"


Your model's input expects normalised tensors. The standard is ImageNet statistics: mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]. But are those the right statistics for Pets? For chest X-rays? For cancer slides? Today: compute the actual per-channel mean and std for 500 random Pets images. Compare to ImageNet. If they differ by more than 0.05 per channel, consider domain-specific normalisation.


### Exercise 2.3 -- Domain normalisation

Compute per-channel mean and std for 500 random Pets images. Compare to ImageNet stats. Is the difference large enough to matter? Run a paired t-test on the red channel means. Report: t-statistic, p-value, decision.


In [ ]:
# Exercise 2.3: Domain normalisation
# -------------------------------------------------------
# YOUR CODE HERE
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the bottom of this notebook.


**Tuesday code scaffold** (read and run, then adapt in the exercise cells above):


In [ ]:
# -- Tuesday: Channel Histograms and Normalisation Parameters (scaffold) --
import numpy as np
from PIL import Image
import torchvision.transforms as T

all_means, all_stds = [], []
for path in sample_paths[:500]:
    img = T.ToTensor()(Image.open(path).convert("RGB")).numpy()
    all_means.append(img.mean(axis=(1,2)))
    all_stds.append(img.std(axis=(1,2)))
mean = np.mean(all_means, axis=0)
std  = np.mean(all_stds,  axis=0)
print(f"Pets mean: {mean.round(3)}  std: {std.round(3)}")
print(f"ImageNet:  [0.485, 0.456, 0.406]  [0.229, 0.224, 0.225]")



---

## WEDNESDAY -- "Augmentation — What the Data Supports"


Augmentation is not free. Horizontal flip is valid for pets (cats are symmetric). It is not valid for chest X-rays (the heart is on the left — flipping creates an anatomically impossible image). Random rotation is valid for microscopy slides (no orientation convention). It is not valid for X-rays (upright vs supine changes the appearance of pleural effusion). Every augmentation requires a clinical or domain justification.


**Wednesday code scaffold** (read and run, then adapt in the exercise cells above):


In [ ]:
# -- Wednesday: Augmentation — What the Data Supports (scaffold) --
import torchvision.transforms as T

# Augmentation audit — apply each to 20 images and visually inspect
augmentations = {
    "HorizontalFlip": T.RandomHorizontalFlip(p=1.0),
    "Rotation30":     T.RandomRotation(30),
    "ColorJitter":    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    "GaussianBlur":   T.GaussianBlur(kernel_size=5),
    "RandomCrop":     T.RandomCrop(180),
}



---

## THURSDAY -- "Label Noise — Are the Annotations Correct?"


The Pets dataset has known annotation inconsistencies — some images are labelled as one breed but visually match another. Model accuracy is bounded above by annotation quality. Today: find the five images your breed classifier (a simple k-NN on colour histograms) disagrees with most strongly. Visually inspect them. Are they mislabelled, ambiguous, or genuinely hard?


### Exercise 2.4 -- Augmentation validity matrix

Build a 5×5 table: augmentation × domain (Pets, X-ray, BreakHis, TCGA, Video). For each cell: Valid / Invalid / Requires care. Justify each Invalid with one sentence.


In [ ]:
# Exercise 2.4: Augmentation validity matrix
# -------------------------------------------------------
# YOUR CODE HERE
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the bottom of this notebook.


**Thursday code scaffold** (read and run, then adapt in the exercise cells above):


In [ ]:
# -- Thursday: Label Noise — Are the Annotations Correct? (scaffold) --
# Find potentially mislabelled images using simple baseline disagreement
from sklearn.neighbors import KNeighborsClassifier

# Use 32-bin colour histogram as features
def colour_hist(path, bins=32):
    img = np.array(Image.open(path).convert("RGB").resize((64,64)))
    return np.concatenate([np.histogram(img[:,:,c], bins=bins, range=(0,255))[0]
                           for c in range(3)]) / (64*64)

# Fit on training set, find training examples with low confidence
# High disagreement = potential label noise or genuinely hard examples



---

## FRIDAY -- "The Friday Build: EDA Report"


Write the EDA report in Pyramid Principle order: findings first, evidence second, methodology last. Fatima needs to know: which classes are hardest, which augmentations are safe, and what the normalisation parameters should be. She does not need to know what a channel histogram is.


**Friday code scaffold** (read and run, then adapt in the exercise cells above):


In [ ]:
# -- Friday: The Friday Build: EDA Report (scaffold) --
# The EDA report must answer three questions:
# 1. Which classes will the model struggle with and why?
# 2. Which augmentations are safe for this domain?
# 3. What normalisation parameters should the pipeline use?
# Write these as three findings before any methodology.


### Friday Workplace Moment

*MediVision AI — Friday standup.*

**Ngozi Eze-Okafor:** Three findings: 8 breeds are underrepresented by more than 50% relative to the median. Horizontal flip is safe; rotation above 15° distorts breed-specific features like ear shape. Pets domain-specific normalisation differs from ImageNet by 0.03 on mean — within tolerance.

**Nurse Folake Balogun:** What does "underrepresented by 50%" mean for a doctor using this model?

**Ngozi Eze-Okafor:** If a patient — I mean a pet — is one of those 8 rare breeds, the model's accuracy may be 15–20 percentage points lower than average. It will appear confident but be wrong more often.

**Nurse Folake Balogun:** That is the same thing that happens with rare conditions in clinical AI. Write that comparison down.

**Dr. Kwame Asante:** That is Week 11's fairness audit. You just wrote the brief for it.



## YOUR WEEK 2 CHECKLIST

Check each item before moving to the next week.
If you cannot explain an item without notes, revisit that section.

- [ ] I know which 8 Pets classes are underrepresented and what that means for per-class accuracy.
- [ ] I have computed domain-specific normalisation parameters and decided whether to use them or ImageNet stats.
- [ ] I have audited 5 augmentations and can justify each choice for the Pets domain.
- [ ] I know why horizontal flip is valid for pet photos but not for chest X-rays.
- [ ] My EDA report leads with findings, not with methodology.


## Challenge Task

> **Core path:** Build a colour histogram baseline classifier (k-NN or logistic regression). Find the 10 images it is most confused about. Are they mislabelled, ambiguous, or a hard subclass? Document each.

> **Advanced path:** Implement a dataset quality score: for each image, compute sharpness (Laplacian variance), brightness (mean luminance), and contrast (std). Flag images in the bottom 5th percentile of each metric as quality concerns. How many does the dataset have?


## Common Mistakes

**Applying horizontal flip to chest X-rays:** The heart is on the left side of the body. A horizontally flipped X-ray is anatomically impossible and trains the model on hallucinated data. Always ask: does this augmentation preserve clinical meaning?

**Using ImageNet normalisation on medical images:** Medical images have different pixel distributions. Compute domain-specific stats. The difference may be small (< 0.05) but cumulative over a training run.

**Reporting class imbalance without clinical context:** "8 classes underrepresented" means nothing to Nurse Folake. "The model may be 15–20% less accurate on rare conditions" is actionable.



---

## Get the Full Book

This notebook is part of **The Computer Vision Internship** -- Book 3 of 9, InternshipInABook(tm).

Covers: natural images, medical X-rays, breast & uterine cancer, video, GradCAM/LIME/SHAP/Integrated Gradients, and fairness audits.

Available at [internshipinabook.com](https://internshipinabook.com)
